# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad-Imran-Toori/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**My lane (locked in ML-02):** Lane 4 — CTR / Engagement Opportunity Scoring.
**My question:** which visible pages under-capture clicks compared with similar pages, and should an editor review first?


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: ranking / scoring** — powered underneath by a supervised **regression**.

My lane asks "which pages first?", not "is this page good or bad?". That wording matters. An editor
cannot review 16,726 pages; they can review a few dozen a week — at 50 a week the backlog is 335
weeks long. So the only output anybody can act on is an **ordered queue**, and the thing I am really
being judged on is what sits at the top of it.
A yes/no classification would hand back thousands of "yes" pages and solve nothing, because it never
answers the actual question — *which* yes goes first.

The scoring works in two steps:

1. **Learn what is normal.** A regression model predicts the CTR a page *like this one* would
   normally get, given what was true about it before anyone clicked: its search position, how much
   volume it gets, its intent, its content type, its freshness. Call that **expected CTR**.
2. **Rank by the gap.** The opportunity score is the **shortfall** — how far the page's actual,
   measured CTR sits *below* that expectation. Biggest shortfall goes to the top of the queue.

So it is a scoring task in shape, a regression in mechanism, and a ranking in how it gets used.

The code below shows why ranking is the right shape: the candidate pool is far larger than any
realistic review capacity, so ordering *is* the product.


In [22]:
# This cell is for CODE (numbers, a query, a check).
import os
import pandas as pd
import numpy as np

CSV_PATH = "data/raw/content_refresh_anonymized.csv"
RAW_URL = ("https://raw.githubusercontent.com/Saad-Imran-Toori/"
           "flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

# Local clone: walk up until the data folder appears.
_start = os.getcwd()
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

# Colab opens only the .ipynb, not the repo -> fetch the file straight from GitHub.
if not os.path.exists(CSV_PATH):
    os.chdir(_start)
    os.makedirs("data/raw", exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(RAW_URL, CSV_PATH)

df = pd.read_csv(CSV_PATH)
print(f"loaded {len(df):,} rows x {df.shape[1]} columns\n")

# My lane only makes sense for pages that are actually VISIBLE in search,
# and only where CTR is stable enough to trust (volume floor).
FLOOR = 500
scope = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= FLOOR)].copy()

print(f"All pages in the slice            : {len(df):,}")
print(f"In scope (position data + >={FLOOR} impr) : {len(scope):,}")

CAPACITY = 50  # pages an editor could realistically review per week
print(f"\nAt ~{CAPACITY} reviews/week, reviewing every in-scope page would take "
      f"{len(scope)/CAPACITY:,.0f} weeks.")
print("-> Capacity is the binding constraint, so the useful output is an ORDER,")
print("   not a yes/no flag. That makes this a RANKING / SCORING task.")


loaded 30,000 rows x 44 columns

All pages in the slice            : 30,000
In scope (position data + >=500 impr) : 16,726

At ~50 reviews/week, reviewing every in-scope page would take 335 weeks.
-> Capacity is the binding constraint, so the useful output is an ORDER,
   not a yes/no flag. That makes this a RANKING / SCORING task.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There are two different quantities here and it matters that I keep them separate.

**What the model predicts (the learning target): `ctr` — an observed measurement.**
`ctr` is computed straight from two counted facts, `clicks_90d / impressions_90d × 100`. Nobody
decided it; a measurement system recorded it. The code below re-derives it from the raw counts and
confirms the two agree to within rounding. That satisfies the rule that matters most in this
framework: **the target must be observed, not defined.**

**What the ranking uses (the score): the shortfall, `expected_ctr − actual_ctr`.**
This is derived, but it is derived from a *learned* expectation plus an *observed* value — not from
a threshold I invented. That is the important difference from my Week-1 draft, where I ranked pages
by "CTR below the median of its position tier". That version made my own arithmetic the target, so
any model trained on it would just be learning to reproduce my rule — a circular result that looks
accurate and discovers nothing. Making expected CTR a *learned* quantity is what breaks the circle.

**The honest gap I have to name.** This dataset contains no observed label for "reviewing this page
turned out to be worth it". Nobody edited a title, logged it, and recorded what happened. So
"deserves review" is a **proxy**, not a measured outcome. I cannot pretend otherwise, and it has two
consequences I carry through the whole project: it shapes how I measure success (Section 3), and it
caps what I am allowed to claim — **decision-support and directional**, never causal.

**Columns that can never be features:** `trend_direction`, `trend_pct` and `is_declining_label` are
the starter pipeline's own label chain, so using them would leak the answer into the inputs.
`content_id` and `client_id` are pseudonyms — good for grouping and splitting, never features.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Check 1: is my learning target OBSERVED, or defined by somebody's rule?
chk = scope.copy()
chk["ctr_recomputed"] = chk["clicks_90d"] / chk["impressions_90d"] * 100
gap = (chk["ctr"] - chk["ctr_recomputed"]).abs()
print("ctr vs clicks/impressions*100 -> max abs difference:", round(gap.max(), 4),
      "(rounding only)")
print("=> ctr is a MEASURED quantity from observed counts, not a rule output.\n")

print(f"ctr median in scope: {scope['ctr'].median():.2f}%   "
      f"(reminder: ctr is a x100 percentage, 0.76 = 0.76%)")

# Check 2: columns I must never use as features (they are the label's source).
BANNED_AS_FEATURES = ["trend_direction", "trend_pct", "is_declining_label"]
print("\nNever features here (rule-derived / label source):", BANNED_AS_FEATURES)
print("IDs (content_id, client_id) are pseudonyms: grouping and splitting only.")


ctr vs clicks/impressions*100 -> max abs difference: 0.005 (rounding only)
=> ctr is a MEASURED quantity from observed counts, not a rule output.

ctr median in scope: 0.17%   (reminder: ctr is a x100 percentage, 0.76 = 0.76%)

Never features here (rule-derived / label source): ['trend_direction', 'trend_pct', 'is_declining_label']
IDs (content_id, client_id) are pseudonyms: grouping and splitting only.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I am naming the metric now, before any model exists, because a definition of "good" written after
seeing results always flatters the results.

**Primary — is the expectation any good? Mean Absolute Error (MAE), in percentage points, on a
client-holdout split.** I group the train/test split by `client_id` (GroupKFold), so the model is
scored on clients it has never seen. That is the honest test: pages inside one client share a
template, a brand and an audience, so a random row split would let the model memorise the client
rather than learn the pattern.

MAE is the right unit here because the shortfall I care about *is* a difference in percentage
points, so the error is measured in exactly the quantity the decision is made in — and unlike RMSE
it is not dominated by a handful of extreme pages.

**The number to beat is computed in the code below, today, with no model at all:** the MAE of the
Week-1 fixed rule (expected CTR = median CTR of the page's own position tier). If a trained model
cannot beat that on held-out clients, the model has not earned its place and I keep the rule and
say so in the paper.

**What the baseline actually scored — and why it is smaller than I expected.** The tier rule lands
at **MAE = 0.185 pp**, against **0.196 pp** for simply predicting one global median for every page.
That is an improvement of only **5.6%**. Knowing a page's position tier, on this slice, barely tells
you more than knowing nothing at all.

I am not going to dress that up, because it cuts in a useful direction: it is direct evidence that
**a position-only rule is too blunt for this lane.** The whole premise of my Week-1 draft was that
position tier defines a fair expectation, and the data says it mostly does not. That is exactly the
gap a multi-signal model exists to fill.

Two honest caveats about the magnitude. CTR in this slice is very low — the in-scope median is
**0.17%** — so every one of these errors is small in absolute terms; what matters is the comparison
between methods, not the size of the number. And a low bar is not automatically an easy win: with so
little spread to exploit, a model that clears 0.185 pp by a trivial margin has not really earned
anything either. I will judge the improvement, not just the direction of it.

**Secondary — is the top of the queue any good? precision@K by manual review**, with K set to real
editor capacity (K = 20, and K = 50). I read the top K pages myself and judge whether each is a
plausible title/snippet/intent review candidate. Overall accuracy across all 16,726 pages is
irrelevant: the queue is consumed from the top down, so only the top matters.

**Later upgrade (Week 5+, on the warehouse):** the full release has ~17 months of daily history, so I
can add a genuinely time-aware check — flag pages from an earlier window, then look at whether their
CTR actually moved in a later window. That is a stronger test than manual review, and it becomes
possible only once I am off this single-snapshot slice.


In [24]:
# This cell is for CODE (numbers, a query, a check).
# The metric must be computable TODAY on a baseline. Here is that baseline:
# the Week-1 fixed rule -> "expected CTR = median CTR of the page's own position tier".
scope["expected_ctr_rule"] = scope.groupby("position_tier")["ctr"].transform("median")

mae_rule   = (scope["ctr"] - scope["expected_ctr_rule"]).abs().mean()
mae_global = (scope["ctr"] - scope["ctr"].median()).abs().mean()

print(f"Baseline A - global median CTR      : MAE = {mae_global:.3f} pp")
print(f"Baseline B - position-tier median   : MAE = {mae_rule:.3f} pp   <- the rule to beat")
print(f"Improvement of the tier rule over global: "
      f"{(mae_global - mae_rule) / mae_global * 100:.1f}%")
print("\n=> Success = a learned expected-CTR model must beat MAE = "
      f"{mae_rule:.3f} pp on HELD-OUT CLIENTS (grouped split on client_id),")
print("   and its top-K queue must beat the rule's top-K on precision@K by manual review.")


Baseline A - global median CTR      : MAE = 0.196 pp
Baseline B - position-tier median   : MAE = 0.185 pp   <- the rule to beat
Improvement of the tier rule over global: 5.6%

=> Success = a learned expected-CTR model must beat MAE = 0.185 pp on HELD-OUT CLIENTS (grouped split on client_id),
   and its top-K queue must beat the rule's top-K on precision@K by manual review.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymised content page**, with all of its metrics aggregated over the same
trailing 90-day window.

My lane's slice is narrower than the full file, for two reasons that are both about not fooling
myself:

- **`avg_position > 0`** — in this dataset `0` does not mean "ranked first", it means *no position
  data at all*. Leaving those rows in would invent a best-possible position for pages nobody
  measured.
- **`impressions_90d >= 500`** — a volume floor. On a page with 40 impressions a single extra click
  swings CTR by 2.5 percentage points, so a "shortfall" down there is noise wearing a lab coat.

The code prints the grain check (no duplicate `content_id`, so one row really is one page), the size
of the in-scope population, and a preview of what the output table will look like once a model
fills in the expected-CTR column.

**The preview exposes a real flaw in my baseline, and it is the most useful thing in this notebook.**
Every page at the top of the rule's queue is a **zero-click page** — real impressions, not one click
— and every one of them scores a shortfall of *exactly* the tier median (0.24 pp for `page_1`).
That is not a coincidence: if actual CTR is 0, then "expected minus actual" is just "expected", so
every zero-click page in a tier receives an identical score. The code below counts how many distinct
scores exist in the rule's top 100, and the answer is **1** — all one hundred pages are handed the
same number. There are 2,530 zero-click pages in scope (15% of the queue), so this is not a rare
edge case.

So the fixed rule **cannot rank the top of its own queue.** It hands the editor a pile of ties and
no way to choose among them — which is a fatal problem for an output whose entire job is ordering.
This is the sharpest argument for the learned expectation: a model conditions on volume, intent,
content type and freshness as well as position, so two zero-click pages get *different* expected
values and therefore a real ordering.

Zero-click pages also need care for a second reason: they have innocent explanations (a brand query
answered directly on the results page, a SERP feature absorbing the click). They will carry their own
reason code and a caveat, rather than a blind spot at the top of the queue.


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Grain check: is one row really one content page?
dupes = scope["content_id"].duplicated().sum()
print(f"duplicate content_id rows in scope: {dupes}  (0 => one row = one page)")
print(f"unit of analysis: {len(scope):,} pages across "
      f"{scope['client_id'].nunique()} pseudonymous clients\n")

# Sketch of the target/output columns this task would produce.
scope["ctr_shortfall_pp"] = scope["expected_ctr_rule"] - scope["ctr"]

cols = ["position_tier", "avg_position", "impressions_90d", "clicks_90d",
        "ctr", "expected_ctr_rule", "ctr_shortfall_pp"]
preview = (scope.sort_values("ctr_shortfall_pp", ascending=False)
                .loc[:, cols]
                .head(8)
                .round(2))
print("Top 8 by CTR shortfall (baseline version of the ranked queue):")
print(preview.to_string(index=False))

# How well does the RULE actually order the top of the queue?
zero_click = int((scope["clicks_90d"] == 0).sum())
top100 = scope.nlargest(100, "ctr_shortfall_pp")
print(f"\nZero-click pages in scope   : {zero_click:,} ({zero_click/len(scope):.0%} of the queue)")
print(f"Distinct shortfall values in the rule's top 100: "
      f"{top100['ctr_shortfall_pp'].nunique()}")
print("=> Every zero-click page scores exactly its tier median, so the rule TIES them")
print("   and cannot say which to review first. A learned per-page expectation")
print("   gives each page its own number, which is what breaks the tie.")
print("\nColumns a model would ADD: expected_ctr_model, ctr_shortfall_model, rank, reason_code.")


duplicate content_id rows in scope: 0  (0 => one row = one page)
unit of analysis: 16,726 pages across 28 pseudonymous clients

Top 8 by CTR shortfall (baseline version of the ranked queue):
position_tier  avg_position  impressions_90d  clicks_90d  ctr  expected_ctr_rule  ctr_shortfall_pp
       page_1           5.5             7737           0  0.0               0.24              0.24
       page_1           6.3             2080           0  0.0               0.24              0.24
       page_1           6.5             2092           0  0.0               0.24              0.24
       page_1           6.8             2845           0  0.0               0.24              0.24
       page_1           3.7              742           0  0.0               0.24              0.24
       page_1           5.7              740           0  0.0               0.24              0.24
       page_1           9.7              599           0  0.0               0.24              0.24
       page_1    

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The fair test is not "is ML fancier" — it is "does a plain rule already do this job". My Week-1 rule
was exactly that if-statement: *expected CTR = the median CTR of your position tier*. One variable,
readable, and honestly not bad. So ML has to earn the upgrade.

**Where the one-variable rule breaks — measured.** It assumes every page at the same position should
attract clicks at the same rate. The code below tests that inside `page_1` (7,064 pages) and the
assumption does not survive: median CTR runs from **0.00%** for informational comparison articles up
to **0.26%** for transactional keyword articles. That is a spread of **0.26 pp inside a single
tier — larger than the tier's own median of 0.24%**. In other words, the variation the rule ignores
is bigger than the signal it uses.

The consequence is not academic. The rule systematically over-flags naturally-low-CTR groups (they
look guilty of a shortfall they never had) and under-flags naturally-high-CTR ones (real
underperformance hides inside a generous tier average). Both failures spend an editor's scarce
review slots on the wrong pages.

Note also the first line of that table: informational comparison articles have a median CTR of
exactly **0.00%** — more than half of them get no clicks at all. A rule built on tier averages has
no way to treat that group differently from the rest of the tier, but any model that sees
`content_type` immediately can.

**What makes it genuinely messy.** Expected CTR depends on several signals *at once* — position
(non-linearly: the drop from 1 to 3 is not the drop from 11 to 13), impressions volume, intent,
content type, keyword competition, freshness. Writing that conditional expectation by hand means
hand-tuning a lookup table across several interacting dimensions, and re-tuning it whenever the mix
shifts. That is precisely the situation where learning the function beats writing it.

**But it has to prove it.** ML earns its place here only if it beats the tier rule's MAE on
held-out clients (Section 3). If it does not, the honest answer is to ship the rule — and that is a
real result worth reporting, not a failure.

**The action this supports.** The output is a ranked review queue with a reason code per page
("high impressions, position 4–10, CTR well under the expectation for this intent and content
type"). A content editor works down it and rewrites the title and meta description, improves snippet
structure, or flags an intent mismatch. Everything I claim stays **observed, directional and
decision-support** — I am prioritising human attention, not proving that any edit causes recovery.


In [26]:
# This cell is for CODE (numbers, a query, a check).
# If CTR only depended on position, then inside ONE position tier every page
# would look alike. Does it? Test on the tier with the most in-scope pages.
top_tier = scope["position_tier"].value_counts().idxmax()
sub = scope[scope["position_tier"] == top_tier].copy()
sub["main_intent"]  = sub["main_intent"].fillna("unknown")
sub["content_type"] = sub["content_type"].fillna("unknown")

grp = (sub.groupby(["content_type", "main_intent"])["ctr"]
          .agg(median_ctr="median", n="size")
          .reset_index())
grp = grp[grp["n"] >= 30].sort_values("median_ctr")

print(f"Position tier examined: '{top_tier}'  ({len(sub):,} pages)\n")
if len(grp) >= 2:
    print("Median CTR (%) INSIDE this one tier, by content type x intent:")
    print(grp.round(2).to_string(index=False))
    lo, hi = grp["median_ctr"].min(), grp["median_ctr"].max()
    tier_med = sub["ctr"].median()
    print(f"\nWeakest sub-group median : {lo:.2f}%")
    print(f"Strongest sub-group median: {hi:.2f}%")
    print(f"Spread inside this one tier: {hi - lo:.2f} pp, "
          f"against a tier median of {tier_med:.2f}% "
          f"({(hi - lo) / tier_med:.0%} of the tier's own expected value).")
    print("=> Position alone does not determine expected CTR. A one-variable rule")
    print("   charges some pages for being the wrong content type, and lets others off.")
else:
    print("Not enough sub-groups above the size floor to compare.")

# How much does the tier rule disagree with itself across sub-groups?
by_type = sub.groupby("content_type")["ctr"].median().round(2)
print("\nMedian CTR by content type within the same tier:")
print(by_type.to_string())


Position tier examined: 'page_1'  (7,064 pages)

Median CTR (%) INSIDE this one tier, by content type x intent:
      content_type   main_intent  median_ctr    n
comparison article informational        0.00   50
   keyword article    commercial        0.22 1210
   keyword article informational        0.24 3962
    feedly article       unknown        0.26   83
   keyword article transactional        0.26 1734

Weakest sub-group median : 0.00%
Strongest sub-group median: 0.26%
Spread inside this one tier: 0.26 pp, against a tier median of 0.24% (108% of the tier's own expected value).
=> Position alone does not determine expected CTR. A one-variable rule
   charges some pages for being the wrong content type, and lets others off.

Median CTR by content type within the same tier:
content_type
comparison article    0.00
feedly article        0.26
keyword article       0.24


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous IDs and aggregates
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Frame in one paragraph.** For a content editor, deciding which visible pages to review this week,
I will build a ranked review queue from the FlyRank content slice, scoring each page by the shortfall
between its learned expected CTR and its observed CTR, measured by MAE on a client-holdout split and
precision@K on the top of the queue. A wrong call costs a wasted review slot (and a pointless rewrite
can damage a working title); a missed one leaves cheap clicks unclaimed. A plain rule is not enough
because expected CTR depends on position, volume, intent and content type at once, and a
position-only rule provably mis-scores whole content types. I will claim observed, directional,
decision-support results only.
